In [ ]:
# ============================================================
# MPRF — Environnement + temps mesurés (réponse R1.12). Google Colab — copier-coller directement
# ============================================================
import sys, os, platform, time
import numpy as np, pandas as pd, scipy, sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score
import warnings; warnings.filterwarnings("ignore")

print("Python      :", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__, "| numpy:", np.__version__, "| pandas:", pd.__version__, "| scipy:", scipy.__version__)
print("OS          :", platform.platform())
cpu = [l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")]
mem = [l for l in open("/proc/meminfo") if l.startswith("MemTotal")][0].split()[1]
print("CPU         :", cpu[0] if cpu else platform.processor(), "| coeurs logiques:", os.cpu_count())
print("RAM         : %.1f Go" % (int(mem) / 1024 / 1024))
try:
    import subprocess
    print("GPU         :", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "aucun (non utilisé)")
except Exception:
    print("GPU         : aucun (non utilisé)")

K_BIRF = 10
ALPHA  = 0.6
# ── Métriques ────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba=None):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean    = np.sqrt(recall_1 * recall_0)

    metrics = {
        "f1_min"   : f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_macro" : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall"   : recall_1,
        "gmean"    : gmean,
    }
    if y_proba is not None:
        metrics["auprc"] = average_precision_score(y_true, y_proba)
    else:
        metrics["auprc"] = np.nan
    return metrics

# ── Vote majoritaire ─────────────────────────────────────────
def ensemble_predict(trees, X, indices):
    preds = np.array([trees[i].predict(X) for i in indices])
    return (preds.sum(axis=0) >= len(indices) / 2).astype(int)

def ensemble_proba(trees, X, indices):
    probas = np.array([trees[i].predict_proba(X)[:, 1] for i in indices])
    return probas.mean(axis=0)

# ── Score combiné pour la sélection ──────────────────────────
def combined_score(y_true, y_pred, alpha):
    """Score(S) = α·F1_min(S) + (1-α)·G-mean(S)"""
    tn = np.sum((y_true == 0) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    f1  = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    gm  = np.sqrt(recall_1 * recall_0)

    return alpha * f1 + (1 - alpha) * gm

# ── MPRF — Forward Selection hill climbing ───────────────────
def MPRF(trees, X_val, y_val, k, fold, alpha=ALPHA):
    """
    Hill climbing forward selection :
    1. Init : un arbre tiré aléatoirement (seed propre au fold)
    2. Critère : Score(S) = α·F1_min(S) + (1-α)·G-mean(S)
    3. À chaque étape : on évalue TOUS les candidats restants
       et on ajoute le meilleur même si égalité (plateau)
    4. best_score mis à jour UNIQUEMENT si amélioration stricte
    5. On s'arrête si tous les candidats dégradent ou si k atteint
    """
    n = len(trees)

    rng_birf = np.random.RandomState(100 + fold)
    init = rng_birf.randint(0, n)

    selected  = [init]
    remaining = list(range(n))
    remaining.remove(init)

    best_score = combined_score(y_val,
                                ensemble_predict(trees, X_val, selected),
                                alpha)

    while len(selected) < k and remaining:

        best_cand       = None
        best_cand_score = -np.inf

        for idx in remaining:
            score = combined_score(y_val,
                                   ensemble_predict(trees, X_val, selected + [idx]),
                                   alpha)
            if score > best_cand_score:
                best_cand_score = score
                best_cand       = idx

        # Avancer si meilleur OU égal (plateau accepté)
        # best_score mis à jour UNIQUEMENT si amélioration stricte
        if best_cand_score >= best_score:
            selected.append(best_cand)
            remaining.remove(best_cand)
            if best_cand_score > best_score:
                best_score = best_cand_score
        else:
            break

    return selected

# ── OBP ──────────────────────────────────────────────────────
def OBP(trees, X_val, y_val, max_size):
    scores = [f1_score(y_val, trees[i].predict(X_val), pos_label=1, zero_division=0)
              for i in range(len(trees))]
    order = np.argsort(scores)[::-1]
    selected, best_score = [], -np.inf
    for idx in order[:max_size]:
        cand = selected + [idx]
        score = f1_score(y_val, ensemble_predict(trees, X_val, cand),
                         pos_label=1, zero_division=0)
        if score >= best_score:
            selected.append(idx)
            best_score = score
    return selected if selected else [order[0]]

# ── IBEP ─────────────────────────────────────────────────────
def IBEP(trees, X_val, y_val, max_size):
    n = len(trees)
    preds   = np.array([trees[i].predict(X_val) for i in range(n)])
    errors  = (preds != y_val).astype(float)
    weights = np.ones(len(y_val)) / len(y_val)
    selected = []
    for _ in range(max_size):
        w_err = errors.dot(weights)
        best  = int(np.argmin(w_err))
        if best in selected:
            break
        selected.append(best)
        weights *= np.exp(errors[best])
        weights /= weights.sum()
    return selected if selected else [0]


df = pd.read_csv("1. risk_factors_cervical_cancer.csv")
df.replace("?", np.nan, inplace=True)
df = df.apply(pd.to_numeric, errors="coerce")
df.fillna(df.median(numeric_only=True), inplace=True)
y = df["Biopsy"].values.astype(int); X = df.drop(columns=["Biopsy"]).values.astype(float)

REP = 20
t_fit, t_sel, t_inf_full, t_inf_sel, sizes = [], [], [], [], []
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
for fold, (tr, te) in enumerate(skf.split(X, y)):
    Xtr, Xte, ytr = X[tr], X[te], y[tr]
    val_size = int(len(tr) * 0.2)
    val_idx = np.random.RandomState(fold).choice(len(ytr), val_size, replace=False)
    sub = np.setdiff1d(np.arange(len(ytr)), val_idx)
    t0 = time.perf_counter()
    rf = RandomForestClassifier(n_estimators=50, class_weight="balanced", random_state=42, n_jobs=1).fit(Xtr[sub], ytr[sub])
    t_fit.append(time.perf_counter() - t0)
    trees = rf.estimators_
    t0 = time.perf_counter()
    sel = MPRF(trees, Xtr[val_idx], ytr[val_idx], K_BIRF, fold, ALPHA)
    t_sel.append(time.perf_counter() - t0)
    allidx = list(range(50)); sizes.append(len(sel))
    t0 = time.perf_counter()
    for _ in range(REP): ensemble_predict(trees, Xte, allidx)
    t_inf_full.append((time.perf_counter() - t0) / REP)
    t0 = time.perf_counter()
    for _ in range(REP): ensemble_predict(trees, Xte, sel)
    t_inf_sel.append((time.perf_counter() - t0) / REP)

ms = lambda a: "%.1f ms (écart-type %.1f)" % (1000 * np.mean(a), 1000 * np.std(a))
print("\nTemps moyens par fold (Cervical Cancer, 1 coeur, n_jobs=1) :")
print("  Entraînement forêt 50 arbres   :", ms(t_fit))
print("  Sélection MPRF (k=10, hors forêt):", ms(t_sel), "-> surcoût relatif : %.1f%% de l'entraînement" % (100 * np.mean(t_sel) / np.mean(t_fit)))
print("  Inférence, 50 arbres (jeu de test):", ms(t_inf_full))
print("  Inférence, arbres sélectionnés    :", ms(t_inf_sel), "| taille moyenne = %.1f arbres" % np.mean(sizes))
print("  Gain d'inférence : x%.1f" % (np.mean(t_inf_full) / np.mean(t_inf_sel)))


Python      : 3.13.18
scikit-learn: 1.6.1 | numpy: 2.4.6 | pandas: 2.2.3 | scipy: 1.16.3
OS          : Linux-6.6.159+-x86_64-with-glibc2.39
CPU         : Intel(R) Xeon(R) CPU @ 2.20GHz | coeurs logiques: 2
RAM         : 12.7 Go
GPU         : aucun (non utilisé)

Temps moyens par fold (Cervical Cancer, 1 coeur, n_jobs=1) :
  Entraînement forêt 50 arbres   : 106.5 ms (écart-type 19.5)
  Sélection MPRF (k=10, hors forêt): 848.0 ms (écart-type 412.5) -> surcoût relatif : 796.3% de l'entraînement
  Inférence, 50 arbres (jeu de test): 11.0 ms (écart-type 2.4)
  Inférence, arbres sélectionnés    : 1.2 ms (écart-type 0.6) | taille moyenne = 5.5 arbres
  Gain d'inférence : x9.0
